# Sign Language Recognition — AI Project 2025-2026
### Antonine University | Faculty of Engineering and Technology

**Project by:** Paul Riachy & Marita Tannoury  
**Presented to:** Dr. Rina Bitar

**Dataset:** Sign Language MNIST (auto-downloaded from Kaggle)  
**Task:** Multi-class Image Classification (ASL letters A–Z, excluding J & Z)  
**ML Models:** Support Vector Machine (SVM), Random Forest  
**DL Models:** Convolutional Neural Network (CNN), Long Short-Term Memory (LSTM)

---

## Quick-Start

1. **Kaggle credentials** — place your `kaggle.json` in `~/.kaggle/` (or upload when prompted in Section 1).
2. Run all cells top-to-bottom (`Runtime → Run all`).
3. After training, use **Section 7** for live webcam prediction.

---

## Section 0 — Install & Import Dependencies

In [ ]:
# ── Install/upgrade required packages ────────────────────────────────────────
import subprocess, sys

pkgs = ["kaggle", "opencv-python-headless"]
for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ Packages ready.")

In [ ]:
import os, time, warnings, io, json, base64, threading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")
print(f"OpenCV     : {cv2.__version__}")

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

---
## Section 1 — Download Dataset from Kaggle

> **One-time setup:** You need a Kaggle API token (`kaggle.json`).  
> Get it at https://www.kaggle.com/settings → **API → Create New Token**.  
> The cell below will auto-detect an existing token or ask you to upload one.

In [ ]:
import kaggle as _kaggle_mod  # noqa – just ensure it imported
from kaggle.api.kaggle_api_extended import KaggleApiExtended

KAGGLE_JSON = os.path.expanduser("~/.kaggle/kaggle.json")
DATA_DIR    = "data"
DATASET     = "datamunge/sign-language-mnist"

# ── 1. Ensure credentials exist ───────────────────────────────────────────
if not os.path.exists(KAGGLE_JSON):
    print("kaggle.json not found — please upload it now.")
    try:
        from google.colab import files as colab_files
        uploaded_creds = colab_files.upload()          # user picks kaggle.json
        for fname, data in uploaded_creds.items():
            os.makedirs(os.path.dirname(KAGGLE_JSON), exist_ok=True)
            with open(KAGGLE_JSON, "wb") as f:
                f.write(data)
        os.chmod(KAGGLE_JSON, 0o600)
        print("✅ kaggle.json saved.")
    except ImportError:
        raise RuntimeError(
            "Not in Colab and ~/.kaggle/kaggle.json is missing.\n"
            "Download your token from https://www.kaggle.com/settings and "
            "place it at ~/.kaggle/kaggle.json"
        )
else:
    print(f"✅ Found credentials at {KAGGLE_JSON}")

# ── 2. Authenticate & download ────────────────────────────────────────────
api = KaggleApiExtended()
api.authenticate()

os.makedirs(DATA_DIR, exist_ok=True)

train_csv = os.path.join(DATA_DIR, "sign_mnist_train.csv")
test_csv  = os.path.join(DATA_DIR, "sign_mnist_test.csv")

if os.path.exists(train_csv) and os.path.exists(test_csv):
    print("📂 Dataset already downloaded — skipping.")
else:
    print(f"⬇️  Downloading '{DATASET}' …")
    api.dataset_download_files(DATASET, path=DATA_DIR, unzip=True)
    print("✅ Download complete.")

# ── 3. Load CSVs ─────────────────────────────────────────────────────────
train_df = pd.read_csv(train_csv)
test_df  = pd.read_csv(test_csv)

print(f"\nTraining set : {train_df.shape[0]:,} samples, {train_df.shape[1]-1} pixel features")
print(f"Test set     : {test_df.shape[0]:,} samples, {test_df.shape[1]-1} pixel features")
print(f"Missing vals — train: {train_df.isnull().sum().sum()} | test: {test_df.isnull().sum().sum()}")

---
## Section 2 — Data Exploration

In [ ]:
# Labels 0-25 → A-Z, but 9 (J) and 25 (Z) are excluded (require motion)
LABEL_MAP = {i: chr(65 + i) for i in range(26) if i not in [9, 25]}

plt.figure(figsize=(14, 4))
counts = train_df['label'].value_counts().sort_index()
counts.index = [LABEL_MAP.get(i, str(i)) for i in counts.index]
sns.barplot(x=counts.index, y=counts.values, palette='viridis')
plt.title('Class Distribution in Training Set', fontsize=14)
plt.xlabel('ASL Letter')
plt.ylabel('Samples')
plt.tight_layout()
plt.show()
print(f"Total classes: {len(counts)} | Samples per class ≈ {int(counts.mean())} (mean)")

In [ ]:
# One sample image per class
unique_labels = sorted(train_df['label'].unique())
n_cols = 6
n_rows = int(np.ceil(len(unique_labels) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 10))
axes = axes.flatten()
for idx, label in enumerate(unique_labels):
    img = train_df[train_df['label'] == label].iloc[0].drop('label').values.reshape(28, 28)
    axes[idx].imshow(img, cmap='gray')
    axes[idx].set_title(LABEL_MAP.get(label, label), fontsize=12)
    axes[idx].axis('off')
for idx in range(len(unique_labels), len(axes)):
    axes[idx].axis('off')
plt.suptitle('Sample ASL Hand Signs (one per class)', fontsize=15)
plt.tight_layout()
plt.show()

---
## Section 3 — Data Preprocessing

In [ ]:
# ── 3.1 Separate features and labels ─────────────────────────────────────
X_train_raw = train_df.drop('label', axis=1).values.astype('float32')
y_train_raw = train_df['label'].values
X_test_raw  = test_df.drop('label', axis=1).values.astype('float32')
y_test_raw  = test_df['label'].values

# ── 3.2 Normalize pixels to [0, 1] ───────────────────────────────────────
X_train_norm = X_train_raw / 255.0
X_test_norm  = X_test_raw  / 255.0

# ── 3.3 Encode labels to consecutive integers 0–23 ────────────────────────
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test  = le.transform(y_test_raw)
NUM_CLASSES = len(le.classes_)
LETTER_LABELS = [LABEL_MAP.get(c, str(c)) for c in le.classes_]

print(f"Classes      : {NUM_CLASSES}")
print(f"Letters      : {LETTER_LABELS}")
print(f"Pixel range  : [{X_train_norm.min():.1f}, {X_train_norm.max():.1f}]")

In [ ]:
# ── 3.4 Reshape for CNN (N,28,28,1) and one-hot encode labels ────────────
X_train_cnn = X_train_norm.reshape(-1, 28, 28, 1)
X_test_cnn  = X_test_norm.reshape(-1, 28, 28, 1)
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

# ── 3.5 Validation split (15 %, stratified) ───────────────────────────────
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_cnn, y_train_cat,
    test_size=0.15, random_state=SEED, stratify=y_train
)
print(f"Train: {X_tr.shape[0]:,} | Val: {X_val.shape[0]:,} | Test: {X_test_cnn.shape[0]:,}")

In [ ]:
# ── 3.6 PCA reduction for ML models (784 → 100 dims) ────────────────────
pca = PCA(n_components=100, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_norm)
X_test_pca  = pca.transform(X_test_norm)
print(f"PCA: 100 components explain "
      f"{pca.explained_variance_ratio_.cumsum()[-1]*100:.1f}% of variance")

---
## Section 4 — Machine Learning Models
### 4.1 Support Vector Machine (SVM)

In [ ]:
print("Training SVM (RBF kernel, C=10) …")
t0 = time.time()
svm_model = SVC(kernel='rbf', C=10, gamma='scale',
                decision_function_shape='ovr', random_state=SEED, probability=True)
svm_model.fit(X_train_pca, y_train)
print(f"Done in {time.time()-t0:.1f}s")

y_pred_svm = svm_model.predict(X_test_pca)
svm_acc = accuracy_score(y_test, y_pred_svm)
svm_f1  = f1_score(y_test, y_pred_svm, average='weighted')
print(f"\nSVM  Accuracy : {svm_acc*100:.2f}%  |  Weighted F1 : {svm_f1*100:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt='d',
            cmap='Blues', xticklabels=LETTER_LABELS, yticklabels=LETTER_LABELS, ax=ax)
ax.set_title('SVM — Confusion Matrix', fontsize=14)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

### 4.2 Random Forest

In [ ]:
print("Training Random Forest (200 trees) …")
t0 = time.time()
rf_model = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_model.fit(X_train_pca, y_train)
print(f"Done in {time.time()-t0:.1f}s")

y_pred_rf = rf_model.predict(X_test_pca)
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1  = f1_score(y_test, y_pred_rf, average='weighted')
print(f"\nRF   Accuracy : {rf_acc*100:.2f}%  |  Weighted F1 : {rf_f1*100:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d',
            cmap='Greens', xticklabels=LETTER_LABELS, yticklabels=LETTER_LABELS, ax=ax)
ax.set_title('Random Forest — Confusion Matrix', fontsize=14)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

---
## Section 5 — Deep Learning Models
### 5.1 Convolutional Neural Network (CNN)

In [ ]:
def build_cnn(num_classes: int) -> keras.Model:
    """Builds a compact but accurate CNN for 28×28 grayscale images."""
    inputs = keras.Input(shape=(28, 28, 1), name="image_input")

    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)   # more stable than Flatten+Dense
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name="predictions")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


cnn_model = build_cnn(NUM_CLASSES)
cnn_model.summary()

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=False   # ASL signs are asymmetric — never flip!
)
datagen.fit(X_tr)

os.makedirs("models", exist_ok=True)
cnn_callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint('models/cnn_best.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=0)
]

cnn_history = cnn_model.fit(
    datagen.flow(X_tr, y_tr, batch_size=64),
    epochs=40,
    validation_data=(X_val, y_val),
    callbacks=cnn_callbacks,
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(cnn_history.history[metric],     label='Train')
    ax.plot(cnn_history.history[f'val_{metric}'], label='Val')
    ax.set_title(f'CNN — {title}'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

y_pred_cnn = np.argmax(cnn_model.predict(X_test_cnn, verbose=0), axis=1)
cnn_acc = accuracy_score(y_test, y_pred_cnn)
cnn_f1  = f1_score(y_test, y_pred_cnn, average='weighted')
print(f"CNN  Accuracy : {cnn_acc*100:.2f}%  |  Weighted F1 : {cnn_f1*100:.2f}%")

# Save final model
cnn_model.save('models/cnn_final.keras')
print("Model saved → models/cnn_final.keras")

### 5.2 LSTM

In [ ]:
# Treat each row of 28 pixels as a time-step → shape (N, 28, 28)
X_train_lstm = X_train_norm.reshape(-1, 28, 28)
X_test_lstm  = X_test_norm.reshape(-1, 28, 28)
X_tr_lstm, X_val_lstm, y_tr_lstm, y_val_lstm = train_test_split(
    X_train_lstm, y_train_cat,
    test_size=0.15, random_state=SEED, stratify=y_train
)

lstm_model = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(64),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
], name='LSTM_Classifier')
lstm_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
lstm_model.summary()

In [ ]:
lstm_history = lstm_model.fit(
    X_tr_lstm, y_tr_lstm,
    batch_size=64, epochs=40,
    validation_data=(X_val_lstm, y_val_lstm),
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=8,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=4, min_lr=1e-6, verbose=1),
    ],
    verbose=1
)

y_pred_lstm = np.argmax(lstm_model.predict(X_test_lstm, verbose=0), axis=1)
lstm_acc = accuracy_score(y_test, y_pred_lstm)
lstm_f1  = f1_score(y_test, y_pred_lstm, average='weighted')
print(f"LSTM Accuracy : {lstm_acc*100:.2f}%  |  Weighted F1 : {lstm_f1*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(lstm_history.history[metric],         label='Train')
    ax.plot(lstm_history.history[f'val_{metric}'], label='Val')
    ax.set_title(f'LSTM — {title}'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## Section 6 — Results Comparison

In [ ]:
results = pd.DataFrame({
    'Model':           ['SVM (RBF)', 'Random Forest', 'CNN', 'LSTM'],
    'Type':            ['ML',        'ML',            'DL',  'DL'],
    'Accuracy (%)':    [round(svm_acc*100, 2), round(rf_acc*100, 2),
                        round(cnn_acc*100, 2), round(lstm_acc*100, 2)],
    'Weighted F1 (%)': [round(svm_f1*100, 2),  round(rf_f1*100, 2),
                        round(cnn_f1*100, 2),  round(lstm_f1*100, 2)],
})
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
for ax, col in zip(axes, ['Accuracy (%)', 'Weighted F1 (%)']):
    bars = ax.bar(results['Model'], results[col], color=colors, edgecolor='white')
    ax.set_title(col, fontsize=13); ax.set_ylim(0, 115)
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
                f'{b.get_height():.1f}%', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Section 7 — Live Camera Sign Detection

> This section works in **two modes** depending on your environment:
>
> | Environment | Mode used |
> |---|---|
> | **Google Colab** | JavaScript webcam → upload image for inference |
> | **Local Jupyter** | OpenCV live video window with real-time prediction |
>
> **Tips for best results:**
> - Use a **plain, light-colored background** (white wall ideal)
> - Keep your hand **centered** in the frame
> - Ensure **even lighting** — avoid strong shadows
> - J and Z are **excluded** (they require motion)

In [ ]:
# ── Shared preprocessing utility ─────────────────────────────────────────
def preprocess_frame(pil_img_or_bgr, is_bgr: bool = False):
    """
    Converts a hand-sign image to a normalized 28×28 array matching
    the Sign Language MNIST format (white hand, black background).

    Args:
        pil_img_or_bgr : PIL Image (RGB/L) OR NumPy BGR frame from cv2.
        is_bgr         : Set True when passing an OpenCV BGR frame.

    Returns:
        normalized (28,28) float32 array,
        thresh     intermediate binary map,
        closed     morphology result,
        resized    28×28 uint8
    """
    if is_bgr:
        gray = cv2.cvtColor(pil_img_or_bgr, cv2.COLOR_BGR2GRAY)
    else:
        gray = np.array(pil_img_or_bgr.convert('L'))

    # Blur to suppress noise
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)

    # Otsu threshold (inverted: hand → white)
    _, thresh = cv2.threshold(
        blurred, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # Auto-inversion: center patch should be mostly hand (white)
    h_img, w_img = thresh.shape
    cy1, cy2 = int(h_img * 0.35), int(h_img * 0.65)
    cx1, cx2 = int(w_img * 0.35), int(w_img * 0.65)
    if thresh[cy1:cy2, cx1:cx2].mean() < 127:
        thresh = cv2.bitwise_not(thresh)

    # Morphological close — fill holes inside the hand
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Largest contour = the hand
    contours, _ = cv2.findContours(
        closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    if contours:
        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        if h > w * 1.4:          # arm present → keep only top (fingers)
            h = int(w * 1.1)
        pad = int(max(w, h) * 0.20)
        x1, y1 = max(0, x - pad), max(0, y - pad)
        x2, y2 = min(closed.shape[1], x + w + pad), min(closed.shape[0], y + h + pad)
        crop = closed[y1:y2, x1:x2]
        ch, cw = crop.shape
        side = max(ch, cw)
        square = np.zeros((side, side), dtype=np.uint8)
        square[(side-ch)//2:(side-ch)//2+ch, (side-cw)//2:(side-cw)//2+cw] = crop
        crop = square
    else:
        crop = closed

    resized    = cv2.resize(crop, (28, 28), interpolation=cv2.INTER_AREA)
    normalized = resized.astype('float32') / 255.0
    return normalized, thresh, closed, resized


def predict_sign(normalized: np.ndarray, top_k: int = 5):
    """Run CNN inference; return list of (letter, confidence%) tuples."""
    probs   = cnn_model.predict(normalized.reshape(1, 28, 28, 1), verbose=0)[0]
    idx     = np.argsort(probs)[::-1][:top_k]
    return [(LABEL_MAP.get(le.classes_[i], '?'), float(probs[i]) * 100) for i in idx]


print("✅ Preprocessing and inference utilities ready.")

In [ ]:
def show_prediction(pil_img, filename='image'):
    """Visualise the full preprocessing pipeline and confidence bars."""
    normalized, thresh, closed, resized = preprocess_frame(pil_img)

    # Preprocessing pipeline visualisation
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
    panels = [
        (np.array(pil_img.convert('L')), 'gray', '① Grayscale'),
        (thresh,  'gray', '② Otsu + auto-flip'),
        (closed,  'gray', '③ Morph. close'),
        (resized, 'gray', '④ Final 28×28'),
    ]
    for ax, (arr, cmap, title) in zip(axes, panels):
        ax.imshow(arr, cmap=cmap); ax.set_title(title, fontsize=10); ax.axis('off')
    fig.suptitle(f'Preprocessing — {filename}', fontsize=12)
    plt.tight_layout(); plt.show()

    # Inference
    preds = predict_sign(normalized)

    # Confidence bar chart
    letters, confs = zip(*preds)
    fig2, ax2 = plt.subplots(figsize=(6, 3))
    colors_bar = ['#a6e3a1', '#89dceb', '#cba6f7', '#f38ba8', '#fab387']
    bars = ax2.barh(list(letters)[::-1], list(confs)[::-1], color=colors_bar)
    ax2.set_xlim(0, 115)
    ax2.set_xlabel('Confidence (%)')
    ax2.set_title(f'Top-5 Predictions — {filename}')
    for bar, conf in zip(bars, list(confs)[::-1]):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{conf:.1f}%', va='center', fontweight='bold', fontsize=10)
    plt.tight_layout(); plt.show()

    print(f"  🥇 {letters[0]:>2}  {confs[0]:5.1f}%  ← prediction")
    for i, (l, c) in enumerate(preds[1:], start=2):
        print(f"  #{i}  {l:>2}  {c:5.1f}%")

### 7A — Google Colab: Webcam Capture (JavaScript)

In [ ]:
# ── Detect environment ────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Colab: {IN_COLAB}")
print("→ Run cell 7A if in Colab, 7B if running locally.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  7A — COLAB WEBCAM CAPTURE                                      ║
# ║  Uses the browser's camera API via JavaScript.                  ║
# ║  Skip this cell if you are running locally (use 7B instead).    ║
# ╚══════════════════════════════════════════════════════════════════╝
if IN_COLAB:
    from IPython.display import display, Javascript
    from google.colab.output import eval_js

    def take_photo(quality=0.92):
        """Open webcam in the browser, capture one frame, return PIL Image."""
        js = Javascript('''
            async function takePhoto(quality) {
                const div = document.createElement('div');
                const capture = document.createElement('button');
                capture.textContent = '📸 Capture';
                capture.style.cssText = `
                    padding:10px 22px; font-size:16px; cursor:pointer;
                    background:#4CAF50; color:#fff; border:none; border-radius:6px;
                    margin-top:8px;`;
                div.appendChild(capture);

                const video = document.createElement('video');
                video.style.cssText = 'display:block; max-width:320px; margin-top:8px;
                    border:3px dashed #aaa; border-radius:8px;';
                video.setAttribute('playsinline','');
                div.appendChild(video);
                document.body.appendChild(div);

                const stream = await navigator.mediaDevices.getUserMedia({video:true});
                video.srcObject = stream;
                await video.play();

                // Resize to match training data aspect ratio
                google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

                await new Promise(resolve => capture.onclick = resolve);

                const canvas = document.createElement('canvas');
                canvas.width  = video.videoWidth;
                canvas.height = video.videoHeight;
                canvas.getContext('2d').drawImage(video, 0, 0);
                stream.getTracks().forEach(t => t.stop());
                div.remove();
                return canvas.toDataURL('image/jpeg', quality);
            }
        ''')
        display(js)
        data_url = eval_js('takePhoto({})'.format(quality))
        binary   = b64decode(data_url.split(',')[1])
        return Image.open(io.BytesIO(binary))

    from base64 import b64decode
    print("📷 Click 'Capture' once your hand is in position …")
    img = take_photo()
    show_prediction(img, filename='webcam_capture')
else:
    print("⚠️  Not in Colab — skip this cell and run 7B instead.")

### 7B — Local Jupyter: Real-Time OpenCV Window

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  7B — LOCAL REAL-TIME CAMERA (OpenCV)                           ║
# ║  Opens a live video window. Press 'Q' to quit.                  ║
# ║  Skip this cell if you are in Colab (use 7A instead).           ║
# ╚══════════════════════════════════════════════════════════════════╝
if not IN_COLAB:
    # ── Config ────────────────────────────────────────────────────
    CAMERA_ID     = 0        # Change if you have multiple cameras
    INFER_EVERY_N = 5        # Run CNN every N frames (reduces lag)
    ROI_X, ROI_Y  = 60, 60  # Top-left of the hand region-of-interest
    ROI_SIZE      = 300      # Square ROI side length in pixels

    cap = cv2.VideoCapture(CAMERA_ID)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open camera {CAMERA_ID}.")

    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    frame_count  = 0
    last_pred    = "—"
    last_conf    = 0.0
    last_top5    = []

    print("📷 Live camera started. Press 'Q' in the window to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)           # Mirror so it feels natural
        h_f, w_f = frame.shape[:2]

        # ── Draw ROI guide box ────────────────────────────────────
        x1 = ROI_X;          y1 = ROI_Y
        x2 = ROI_X+ROI_SIZE; y2 = ROI_Y+ROI_SIZE
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, 'Place hand here', (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        # ── Inference every N frames ──────────────────────────────
        frame_count += 1
        if frame_count % INFER_EVERY_N == 0:
            roi = frame[y1:y2, x1:x2]
            if roi.size > 0:
                normalized, _, _, _ = preprocess_frame(roi, is_bgr=True)
                last_top5 = predict_sign(normalized, top_k=5)
                last_pred, last_conf = last_top5[0]

        # ── Overlay prediction ────────────────────────────────────
        label_txt = f"{last_pred}  {last_conf:.1f}%"
        cv2.putText(frame, label_txt, (x1, y2 + 35),
                    cv2.FONT_HERSHEY_DUPLEX, 1.4, (0, 200, 255), 3)

        # ── Mini top-5 sidebar ────────────────────────────────────
        bar_x = x2 + 20
        cv2.putText(frame, 'Top-5', (bar_x, y1 + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        for rank, (ltr, conf) in enumerate(last_top5):
            by = y1 + 50 + rank * 36
            bar_len = int(conf * 1.2)  # max ~120 px for 100%
            bar_col = (0, 220, 130) if rank == 0 else (180, 180, 180)
            cv2.rectangle(frame, (bar_x, by), (bar_x + bar_len, by + 22), bar_col, -1)
            cv2.putText(frame, f'{ltr} {conf:.0f}%', (bar_x + bar_len + 4, by + 17),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

        cv2.putText(frame, 'Q: quit', (10, h_f - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

        cv2.imshow('ASL Sign Language — Live Recognition', frame)

        if cv2.waitKey(1) & 0xFF in (ord('q'), ord('Q'), 27):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("\n📷 Camera closed.")
else:
    print("⚠️  In Colab — skip this cell and run 7A instead.")

### 7C — Upload Image (works everywhere)

In [ ]:
# ── Upload one or more hand-sign photos and predict ───────────────────────
if IN_COLAB:
    from google.colab import files as colab_files
    print('📂 Select one or more hand-sign images …')
    uploaded_imgs = colab_files.upload()
    for fname, data in uploaded_imgs.items():
        print(f'\n── {fname} ──')
        show_prediction(Image.open(io.BytesIO(data)), filename=fname)
else:
    # Local: place images in an 'images/' folder and list them
    import glob
    img_paths = glob.glob('images/*.jpg') + glob.glob('images/*.png')
    if img_paths:
        for p in img_paths:
            print(f'\n── {p} ──')
            show_prediction(Image.open(p), filename=os.path.basename(p))
    else:
        print("No images found in ./images/ — place some .jpg/.png files there,"
              " or capture via the live camera (7B).")

---
## Section 8 — Summary

**Key takeaways:**
- **CNN** achieves the highest accuracy thanks to spatial feature learning via convolutional filters.
- **SVM (RBF)** is the strongest classical ML model when paired with PCA.
- **LSTM** reasons over image rows as sequences, competitive but slower than CNN.
- **Random Forest** is fast to train and interpretable, at the cost of some accuracy.
- The live demo confirms real-world applicability of the trained CNN.

**AI tools used:** Claude (Anthropic)  
**Dataset:** [Sign Language MNIST — Kaggle](https://www.kaggle.com/datasets/datamunge/sign-language-mnist)